In [ ]:
!pip install -U transformers datasets rouge-score accelerate tensorboard

import torch
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from datasets import load_dataset
from rouge_score import rouge_scorer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 15.0 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=92715fd0bd86ded0b28de676e617b9b042a768507f7f27aad89c61bdf5c06266
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.19.0
    Uninstalling t

In [ ]:
!pwd

/content


In [105]:
# ======================
# Config
# ======================

MODEL_CHECKPOINT = "google-t5/t5-small"   # or t5-base, t5-large
MAX_INPUT_LENGTH = 2048
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 8
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
OUTPUT_DIR = "./t5-summarization-model"
CSV_PATH = "tldr.csv"

prefix = "summarize: "   # or "slangify: " etc.

# ======================
# Load dataset from single CSV and split
# ======================

data_files = {"data": CSV_PATH}
dataset = load_dataset("csv", data_files=data_files)
dataset["data"] = dataset["data"].select(range(300))

# 90% train, 10% validation
dataset = dataset["data"].train_test_split(test_size=0.1, seed=42)

dataset["validation"] = dataset["test"]
del dataset["test"]

# ======================
# Tokenizer & model
# ======================

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

{'prompt': "SUBREDDIT: r/relationships TITLE: I (f/22) have to figure out if I want to still know these girls or not and would hate to sound insulting POST: Not sure if this belongs here but it's worth a try. Backstory: When I (f/22) went through my first real breakup 2 years ago because he needed space after a year of dating roand it effected me more than I thought. It was a horrible time in my life due to living with my mother and finally having the chance to cut her out of my life. I can admit because of it was an emotional wreck and this guy was stable and didn't know how to deal with me. We ended by him avoiding for a month or so after going to a festival with my friends. When I think back I wish he just ended. So after he ended it added my depression I suffered but my friends helped me through it and I got rid of everything from him along with cutting contact. Now: Its been almost 3 years now and I've gotten better after counselling and mild anti depressants. My mother has been o

In [106]:
dataset['train'][0]

{'prompt': "SUBREDDIT: r/AskReddit TITLE: Reddit, a nursing manager made my girlfriend cry. Help me plot Revenge! POST: To make a long story short, my girlfriend who was new to the medicine floor, left her coffee mug in an area that she shouldn't have, and the nursing manager threw it away. To be fair, people leave thermoses and coffee cups in that area all the time without problem. AND, this was a $30 super nice vacuum-insulated mug that I bought as a bday gift. AND, the nursing manager threw away everyone's items/mugs without telling anyone, while they were standing not 10 feet away doing rounds (she was in a back room so they couldnt see what she was doing, and weren't really paying attention). Nobody ever explained that you cannot leave items in that area, and when questioned the manager yelled at my girlfriend for not knowing the rules (as I said, they were never explained) and threatened to call the hospital CMO. TL;DR:",
 'completion': 'HELP ME PLOT THE PERFECT REVENGE. I have n

In [107]:
# ======================
# Preprocessing
# ======================

def preprocess_function(examples):
    """
    Uses:
      - input:  examples["prompt"], examples["completion"]
      - target: examples["GenZ_completion"]
    """

    if None in examples["GenZ_completion"]:
      return

    inputs = [prefix + doc for doc in examples["prompt"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        # padding="max_length"
    )

    # Targets
    targets = examples["GenZ_completion"]

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
            # padding="max_length"
        )


    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=dataset["train"].column_names
)

Map (num_proc=4):   0%|          | 0/270 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your 

Map (num_proc=4):   0%|          | 0/30 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your 

In [108]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 270
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 30
    })
})

In [109]:
# ======================
# Metrics (ROUGE)
# ======================

rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"], use_stemmer=True
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decode predictions
    pred_ids = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)

    # Replace -100 in labels as well
    labels_ids = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)

    rouge1_list, rouge2_list, rougeL_list = [], [], []
    for pred, label in zip(decoded_preds, decoded_labels):
        scores = rouge.score(label, pred)
        rouge1_list.append(scores["rouge1"].fmeasure)
        rouge2_list.append(scores["rouge2"].fmeasure)
        rougeL_list.append(scores["rougeL"].fmeasure)

    return {
        "rouge1": np.mean(rouge1_list),
        "rouge2": np.mean(rouge2_list),
        "rougeL": np.mean(rougeL_list),
    }

In [110]:
# ======================
# Training setup
# ======================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    # evaluation_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=100,
    save_steps=500,
    eval_steps=500,
    save_total_limit=3,
    # load_best_model_at_end=True,
    metric_for_best_model="rouge1",
    greater_is_better=True,
    fp16=True,
    # predict_with_generate=True,
    report_to="tensorboard",
    seed=42,
    remove_unused_columns=False
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    return_tensors="pt"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

/tmp/ipython-input-2474598618.py:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [111]:
# ======================
# Train
# ======================

history = trainer.train()

# Save final model & tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Training complete! Model saved to {OUTPUT_DIR}")

Step,Training Loss
100,3.931200


Training complete! Model saved to ./t5-summarization-model


In [112]:
# ======================
# Inference example
# ======================

fine_tuned_model = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR)
fine_tuned_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

fine_tuned_model.to(device)

# Example prompt (same style as your 'prompt' column)
input_text = """
SUBREDDIT: r/Advice TITLE: How to help GF's parents make friends so she'll feel comfortable moving out? POST: My (26F) gf and I (25M) have been dating for close to 6 years. The first two while we were in uni, 2 years of us basically trying to figure out life (job hunting, switching etc), and 2 years of having a lot more things figured out on the path to settlement. For me, 2 of those years was living near campus, 2 years at home with the parents, and then 2 years away from home (1 year about an hour away from the GF, 1 year and currently a 5 hour drive from GF). For her, she has always lived with her parents.The university was close enough that she didn't need to move out. Now, that isn't to say she isn't independent. She takes care of all the finances, shopping, housekeeping etc at home, but her parents are in good physical health to do this on their own. So here's where the problem is. I am living quite a bit ways away, but willing to move closer back (sort of giving up a job I love, though might get laid off soon) if her and i moved in together. She'll often mention how she wants to do it and talk about what it would be like. But when I get serious about it, she always brings up how her parents would be lonely and depressed if she wasn't there. Fair enough (I argued we'd move at least an hour away from them so that the distance wouldn't be insane, but no bite) So now, I'm wondering, with parents that are ~45-55ish age range, and Indian in a community that is predominately Canadian, how can I go about helping her parents make friends? TL;DR:
"""

inputs = fine_tuned_tokenizer(
    prefix + input_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
).to(device)

summary_ids = fine_tuned_model.generate(
    inputs["input_ids"],
    max_length=128,
    min_length=20,
    num_beams=4,
    early_stopping=True,
    no_repeat_ngram_size=2
)

output_text = fine_tuned_tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)
print(f"Model output: {output_text}")

Model output: my (26F) gf and I (25M) have been dating for close to 6 years. the first two were in uni, 2 years of us basically trying to figure out life (job hunting, switching etc) for her, she has always lived with her parents, but she's willing to move closer back (sort of giving up a job I love, though might get laid off soon)
